<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [13]</a>'.</span>

In [1]:
# %load_ext autoreload
# %autoreload 2

In [2]:
from moseplib.data import pointcloud_processing, timeseries_processing, config, pc_statistics, utils
from mosep_analysis.data.config import (
    TARGET_EXTENTS_VIF,
    TARGET_EXTENTS_VIF_SPLITS,
    INTERIM_DATA_FOLDER,
    PROCESSED_DATA_FOLDER,
    ROOF_EXTENT,
    TARGET_DISTANCES,
)
from mosep_analysis.visualization.timeseries import (
    plot_timeseries_separate_axes,
    target_stat_vs_precipitation,
    histograms_targets_attime,
    histogram_targets_interactive,
)
from moseplib.data.utils import flatten_multiindex

import pandas as pd
from pointcloudset import PointCloud

from pathlib import Path
import pickle
import warnings

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import panel as pn

pn.extension("plotly")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Load Met Data from Bagfile

In [3]:
BAG_NAME: str = "molisens_met_2023_08_07-15_36_45_converted"
DATA_DIR: str = str(INTERIM_DATA_FOLDER / "ViF_Roof" / "data")
TIMESTAMPS_HISTOGRAMS: list = ["2023-08-07T13:37:00"]

In [4]:
# Parameters
BAG_NAME = "molisens_met_2023_08_28-21_02_52_converted"
DATA_DIR = "/datalocal/chg/MOLISENSext/ViF_Roof/data"
TIMESTAMPS_HISTOGRAMS = []


In [5]:
# Todo: Check if this code is still up to date (shift, rounding, etc.)

DATA_DIR = Path(DATA_DIR)

path = PROCESSED_DATA_FOLDER / f"weather_df_{BAG_NAME}.pickle"
if not path.exists():
    df_ws100 = timeseries_processing.load(
        DATA_DIR / BAG_NAME,
        "/sensing/aws/ws100_measurements",
        config.PATH_TO_LUFFT_MSGS,
        timestamp_source="header",
    )
    df_ws501 = timeseries_processing.load(
        DATA_DIR / BAG_NAME,
        "/sensing/aws/ws501_measurements",
        config.PATH_TO_LUFFT_MSGS,
        timestamp_source="header",
    )

    # Unify timestamp to concat the dataframes
    # Set the ferquency of the index to 1s
    df_ws100.index.asfreq = "S"
    df_ws501.index.asfreq = "S"

    # and round the index to the nearest second for easier analysis
    df_ws100.index = df_ws100.index.round("S")
    df_ws501.index = df_ws501.index.round("S")

    # Shift the intensity of the precipitation to the past by 60s to match the rest of the data.
    df_ws100.loc[:, ("precipitation", "intensity_hour_shifted")] = df_ws100.precipitation.intensity_hour.shift(
        periods=-60, freq="S"
    )
    df_ws100.loc[:, ("precipitation", "intensity_hour_shifted")] = df_ws100.loc[
        :, ("precipitation", "intensity_hour_shifted")
    ].fillna(0.0)

    # Align dataframes in time and concat
    if df_ws100.index.equals(df_ws501.index) is False:
        print("Warning: Dataframes have different timestamps, aligning them.")
        df_ws100, df_ws501 = df_ws100.align(df_ws501, join="inner", axis=0)
    df = pd.concat([df_ws100, df_ws501], axis=1)
    # remove lines with nan
    if df.isna().sum().max() <= 1:
        df = df.dropna()
    else:
        raise ValueError("Too many nan values in the dataframe")

    # Save the df to pickle
    with open(path, "wb") as handle:
        pickle.dump(df, handle, protocol=pickle.HIGHEST_PROTOCOL)

else:
    # load from pickle
    with open(path, "rb") as handle:
        df = pickle.load(handle)

### Met Data Overview

In [6]:
parameters = {
    "precipitation intensity": {
        "data_key": ("precipitation", "intensity_hour_shifted"),
        "color": "black",
        "axis_title": "Precipitation Intensity (mm/h)",
    },
    "wind speed": {
        "data_key": ("wind", "speed_avg"),
        "color": "blue",
        "axis_title": "Wind Speed (m/s)",
    },
    "temperature": {
        "data_key": ("temperature", "average"),
        "color": "red",
        "axis_title": "Temperature (°C)",
    },
    "humidity": {
        "data_key": ("humidity", "relative_avg"),
        "color": "green",
        "axis_title": "Humidity (%)",
    },
    "pressure": {
        "data_key": ("pressure", "relative_avg"),
        "color": "purple",
        "axis_title": "Pressure (hPa)",
    },
    "radiation": {
        "data_key": ("radiation", "current"),
        "color": "orange",
        "axis_title": "Radiation (W/m²)",
    },
}

plot_timeseries_separate_axes(df, parameters)

### Find the most relevant precipitation parameters for further analysis:

In [7]:
df.precipitation.loc[:, ["absolute", "differential", "total_precipitation_particles", "total_drops"]].corrwith(
    df.precipitation.intensity_hour_shifted
)

Parameter
absolute                         0.224701
differential                     0.258109
total_precipitation_particles    0.121926
total_drops                      0.276986
dtype: float64

In [8]:
fig1 = px.line(df.precipitation.absolute)
fig1.update_traces(line=dict(color="blue"), connectgaps=True)
fig2 = px.line(df.precipitation.differential.resample("10s").sum() * 60)
fig2.update_traces(line=dict(color="red"))
fig3 = px.line(df.precipitation.intensity_hour_shifted)
fig3.update_traces(line=dict(color="green"), connectgaps=True)
# fig6 = px.line(df.precipitation.intensity_hour)
# fig6.update_traces(line=dict(color="lightgreen"))
fig4 = px.line(df.precipitation.total_precipitation_particles.resample("10s").sum())
fig4.update_traces(line=dict(color="gold"))
fig5 = px.line(df.precipitation.total_drops.resample("10s").sum())
fig5.update_traces(line=dict(color="black"))
# fig6 = px.line(df.

# Create a subplot with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add the traces from fig1, fig3, fig4, and fig5 to the primary y-axis
for trace in fig1.data + fig3.data + fig4.data + fig5.data:
    fig.add_trace(trace, secondary_y=False)

# Add the traces from fig2 to the secondary y-axis
for trace in fig2.data:
    fig.add_trace(trace, secondary_y=True)

# Update the layout
fig.update_layout(width=1400, height=600)

fig.show()

From the correlations with `intensity_hour_shifted` and the plot above the following precipitation parameters seem are chosen for further analysis.

In [9]:
RESAMPLE_FREQ = "1min"
columns_and_aggregation = {
    ("precipitation", "intensity_hour_shifted"): "first",
    ("precipitation", "intensity_hour"): "first",
    ("precipitation", "differential"): "sum",
    ("precipitation", "total_precipitation_particles"): "sum",
    ("precipitation", "total_drops"): "sum",
    ("wind", "speed_avg"): "mean",
    ("temperature", "average"): "mean",
    ("humidity", "relative_avg"): "mean",
    ("pressure", "relative_avg"): "mean",
    ("radiation", "current"): "mean",
}

df_relevant = df.loc[:, columns_and_aggregation.keys()]

columns = flatten_multiindex(df_relevant)
df_relevant = df_relevant.resample(RESAMPLE_FREQ).agg({".".join(k): v for k, v in columns_and_aggregation.items()})

In [10]:
# color df by correlation+
df_relevant.corr()
fig = go.Figure(data=go.Heatmap(z=df_relevant.corr(), x=df_relevant.columns, y=df_relevant.columns))
fig.show()

df_relevant.columns = columns_and_aggregation
df_relevant.corr()

precipitation  \
                                            intensity_hour_shifted   
precipitation intensity_hour_shifted                      1.000000   
              intensity_hour                              0.673517   
              differential                                0.952825   
              total_precipitation_particles               0.556115   
              total_drops                                 0.819786   
wind          speed_avg                                  -0.207275   
temperature   average                                    -0.246458   
humidity      relative_avg                                0.267950   
pressure      relative_avg                                0.356367   
radiation     current                                     0.206583   

                                                                         \
                                            intensity_hour differential   
precipitation intensity_hour_shifted              0.673517     0.952825   
              intensity_hour                      1.000000     0.613656   
              differential                        0.613656     1.000000   
              total_precipitation_particles       0.516855     0.613969   
              total_drops                         0.747716     0.840173   
wind          speed_avg                          -0.234845    -0.221423   
temperature   average                            -0.324988    -0.263611   
humidity      relative_avg                        0.332363     0.283987   
pressure      relative_avg                        0.367573     0.359640   
radiation     current                             0.292822     0.164493   

                                                                           \
                                            total_precipitation_particles   
precipitation intensity_hour_shifted                             0.556115   
              intensity_hour                                     0.516855   
              differential                                       0.613969   
              total_precipitation_particles                      1.000000   
              total_drops                                        0.593184   
wind          speed_avg                                         -0.220978   
temperature   average                                           -0.298028   
humidity      relative_avg                                       0.304467   
pressure      relative_avg                                       0.269241   
radiation     current                                            0.004339   

                                                             wind temperature  \
                                            total_drops speed_avg     average   
precipitation intensity_hour_shifted           0.819786 -0.207275   -0.246458   
              intensity_hour                   0.747716 -0.234845   -0.324988   
              differential                     0.840173 -0.221423   -0.263611   
              total_precipitation_particles    0.593184 -0.220978   -0.298028   
              total_drops                      1.000000 -0.435073   -0.511032   
wind          speed_avg                       -0.435073  1.000000    0.893948   
temperature   average                         -0.511032  0.893948    1.000000   
humidity      relative_avg                     0.523432 -0.899101   -0.998614   
pressure      relative_avg                     0.567093 -0.969904   -0.847285   
radiation     current                          0.198316  0.035837    0.115839   

                                                humidity     pressure  \
                                            relative_avg relative_avg   
precipitation intensity_hour_shifted            0.267950     0.356367   
              intensity_hour                    0.332363     0.367573   
              differential                      0.283987     0.359640   
              total_precipitation_particles     

## Load PC data

In [11]:
TOPICS = {
    "lid_pts": "/sensing/lidar/points",
    "lid_pts2": "/sensing/lidar/points2",
    "rad_pts": "/sensing/radar/points",
}

In [12]:
dataset_rain = pointcloud_processing.load_pointcloudset(
    DATA_DIR, BAG_NAME, topic=TOPICS["lid_pts"], verbose=True, invert_axes=["x", "y"]
)

Searching for pointcloudset files in:
/datalocal/chg/MOLISENSext/ViF_Roof/data/pointcloudset/sensing_lidar_points/molisens_met_2023_08_28-21_02_52_conver
ted

Dataset loaded from:
/datalocal/chg/MOLISENSext/ViF_Roof/data/pointcloudset/sensing_lidar_points/molisens_met_2023_08_28-21_02_52_conver
ted

 start =    2023-08-28 21:02:55.193490
 end =      2023-08-28 21:35:44.234713
 duration = 0:32:49.041223
 length =   19618
 avg frequency =  9.96 Hz

Inverting axes: ['x', 'y']


### Resample Dataset to 1min

<font color='red'>For now</font>, we will resample the dataset to 1min resolution due to long computation times. Once we have a better understanding of the data, we can adjust the resolution accordingly.

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [13]:
# Remove the first 20 seconds of the dataset. This should be done when loading data !!!!
path = PROCESSED_DATA_FOLDER / f"rain_minutes_{BAG_NAME}.pickle"
if not path.exists():
    ds_minutes = pointcloud_processing.resample_dataset(
        dataset_rain[20 * 10 :], "1min", extra_statistics=["std", "sum"]
    )

    # Save the resampled datasets to pickle
    with open(path, "wb") as handle:
        pickle.dump(ds_minutes, handle, protocol=pickle.HIGHEST_PROTOCOL)
else:
    # load from pickle
    with open(path, "rb") as handle:
        ds_minutes = pickle.load(handle)

  0%|                                                                                                               | 0/33 [00:00<?, ?it/s]

  3%|███                                                                                                 | 1/33 [02:30<1:20:19, 150.60s/it]

  6%|██████                                                                                              | 2/33 [05:48<1:32:02, 178.15s/it]

  9%|█████████                                                                                           | 3/33 [09:06<1:33:38, 187.27s/it]

 12%|████████████                                                                                        | 4/33 [12:17<1:31:18, 188.91s/it]

...

 94%|█████████████████████████████████████████████████████████████████████████████████████████████      | 31/33 [1:37:59<06:27, 193.50s/it]

ignoring exception in ensure_cleanup_on_exception
Traceback (most recent call last):
  File "/home/chg/mosep-analysis/.venv/lib/python3.11/site-packages/dask/dataframe/shuffle.py", line 226, in ensure_cleanup_on_exception
    yield
  File "/home/chg/mosep-analysis/.venv/lib/python3.11/site-packages/dask/dataframe/dask_expr/_shuffle.py", line 507, in _shuffle_group
    p.append(d, fsync=True)
  File "/home/chg/mosep-analysis/.venv/lib/python3.11/site-packages/partd/encode.py", line 25, in append
    self.partd.append(data, **kwargs)
  File "/home/chg/mosep-analysis/.venv/lib/python3.11/site-packages/partd/buffer.py", line 45, in append
    self.flush(keys)
  File "/home/chg/mosep-analysis/.venv/lib/python3.11/site-packages/partd/buffer.py", line 99, in flush
    self.slow.append(dict(zip(keys, self.fast.get(keys))))
  File "/home/chg/mosep-analysis/.venv/lib/python3.11/site-packages/partd/file.py", line 42, in append
    f.write(v)
OSError: [Errno 28] No space left on device

During han

 94%|█████████████████████████████████████████████████████████████████████████████████████████████      | 31/33 [1:40:36<06:29, 194.72s/it]


ignoring exception in ensure_cleanup_on_exception
Traceback (most recent call last):
  File "/home/chg/mosep-analysis/.venv/lib/python3.11/site-packages/dask/dataframe/shuffle.py", line 226, in ensure_cleanup_on_exception
    yield
  File "/home/chg/mosep-analysis/.venv/lib/python3.11/site-packages/dask/dataframe/dask_expr/_shuffle.py", line 507, in _shuffle_group
    p.append(d, fsync=True)
  File "/home/chg/mosep-analysis/.venv/lib/python3.11/site-packages/partd/encode.py", line 25, in append
    self.partd.append(data, **kwargs)
  File "/home/chg/mosep-analysis/.venv/lib/python3.11/site-packages/partd/buffer.py", line 45, in append
    self.flush(keys)
  File "/home/chg/mosep-analysis/.venv/lib/python3.11/site-packages/partd/buffer.py", line 99, in flush
    self.slow.append(dict(zip(keys, self.fast.get(keys))))
  File "/home/chg/mosep-analysis/.venv/lib/python3.11/site-packages/partd/file.py", line 42, in append
    f.write(v)
OSError: [Errno 28] No space left on device

During ha

OSError: [Errno 28] No space left on device

## Exploratory Data Analysis (EDA)

### Setup

In [ ]:
TIME_AGGREGATION = "mean"
SPACE_AGGREGATION = pc_statistics.mean_intensity
# SPACE_AGGREGATION = pc_statistics.sum_intensity

In [ ]:
target_statistics = pointcloud_processing.subset_and_aggregate_dataset(
    dataset=ds_minutes[TIME_AGGREGATION],
    splits=TARGET_EXTENTS_VIF,
    agg_func=SPACE_AGGREGATION,
    return_type="df",
)
target_statistics.columns.name = "target"

In [ ]:
subtarget_statistics = pointcloud_processing.subset_and_aggregate_dataset(
    dataset=ds_minutes[TIME_AGGREGATION],
    splits=TARGET_EXTENTS_VIF_SPLITS,
    agg_func=SPACE_AGGREGATION,
    return_type="df",
)
subtarget_statistics.columns.names = ["target", "color"]

#### Normalize Data

In [ ]:
NORM_KIND = "standard"

target_statistics_norm = utils.normalize_df(target_statistics, kind=NORM_KIND)
subtarget_statistics_norm = utils.normalize_df(subtarget_statistics, kind=NORM_KIND)
df_relevant_norm = utils.normalize_df(df_relevant, kind=NORM_KIND, fillna=True)

### Mean intensity of full target (all reflectivities)

In [ ]:
NORMALIZE = False

if NORMALIZE:
    targ_stats = target_statistics_norm
    weather_stats = df_relevant_norm
else:
    targ_stats = target_statistics
    weather_stats = df_relevant


fig = target_stat_vs_precipitation(
    targ_stats,
    weather_stats.precipitation.intensity_hour_shifted,
    exclude_cols=None,  # ["Target-02"],
    fig_size=(1000, 500),
    precipitation_name="Rainfall Rate",
    yaxis_title="Mean Intensity",
    yaxis2_title="Rainfall Rate [mm/h]",
)
fig.show()
fig = target_stat_vs_precipitation(
    targ_stats,
    weather_stats.wind.speed_avg,
    exclude_cols=None,  # ["Target-02"],
    fig_size=(1000, 500),
    precipitation_name="Wind Speed",
    yaxis_title="Mean Intensity",
    yaxis2_title="Wind Speed [m/s]",
)
fig.show()
fig = target_stat_vs_precipitation(
    targ_stats,
    weather_stats.radiation.current,
    exclude_cols=None,  # ["Target-02"],
    fig_size=(1000, 500),
    precipitation_name="Radiation",
    yaxis_title="Mean Intensity",
    yaxis2_title="Radiation [W/m²]",
)
fig.show()

### Mean intensity of targets spearate by intensity

In [ ]:
NORMALIZE = True

if NORMALIZE:
    subtarg_stats = subtarget_statistics_norm
    weather_stats = df_relevant_norm
    title_addon = "Normalized "
else:
    subtarg_stats = subtarget_statistics
    weather_stats = df_relevant
    title_addon = ""

target_stat_vs_precipitation(
    subtarg_stats,
    weather_stats.precipitation.intensity_hour_shifted,
    value_name="intensity",
    precipitation_name="Rainfall Rate",
    yaxis_title=f"{title_addon}Mean Intensities",
    yaxis2_title=f"{title_addon}Rainfall Rate",
    compress_legend=False,
    fig_size=(1200, 600),
)

In [ ]:
# timestamp = "2023-08-29T04:37:00"

# ROOF_EXTENT.apply_limits(utils.get_pointcloud_from_timestamp(ds_minutes["mean"], timestamp)).plot(
#     color="intensity",
#     hover_data=["intensity"],
#     overlay={
#         "Target-05": TARGET_EXTENTS_VIF["Target-05"].apply_limits(
#             utils.get_pointcloud_from_timestamp(ds_minutes["mean"], timestamp)
#         )
#     },
# )

### Histograms for each target

https://plotly.com/chart-studio-help/histogram/#normalizing-a-histogram

In [ ]:
plot_data = subtarget_statistics.loc[:, pd.IndexSlice[:, "grey"]]
plot_data = plot_data.melt(value_name="intensity")

fig = px.histogram(
    plot_data,
    x="intensity",
    color="target",
    marginal="box",  # can be 'rug', `box`, `violin`
    hover_data=plot_data.columns,
    histfunc="count",  # can be "avg"
    nbins=100,
    histnorm="percent",  # '', 'percent', 'probability', 'density', 'probability density'
    title="Intensity Distribution of Targets (aggregated in space) over time",
    # log_y=True,
)
fig.show()

#### Range

In [ ]:
# target_pcs = pointcloud_processing.subset_and_aggregate_dataset(ds_minutes["mean"], TARGET_EXTENTS_VIF_SPLITS)
# app = histogram_targets_interactive(target_pcs)
# app.run_server(jupyter_mode="external")

In [ ]:
TARGET = "Target-02"
VALUE = "range"

if TIMESTAMPS_HISTOGRAMS:
    target_pcs = pointcloud_processing.subset_and_aggregate_dataset(ds_minutes["mean"], TARGET_EXTENTS_VIF_SPLITS)
    fig = histograms_targets_attime(
        target_pcs,
        df_relevant_norm,
        TIMESTAMPS_HISTOGRAMS,
        TARGET,
        VALUE,
    )
    fig.show()

#### Intensity

In [ ]:
TARGET = "Target-02"
VALUE = "intensity"

if TIMESTAMPS_HISTOGRAMS:
    target_pcs = pointcloud_processing.subset_and_aggregate_dataset(ds_minutes["mean"], TARGET_EXTENTS_VIF_SPLITS)
    fig = histograms_targets_attime(
        target_pcs,
        df_relevant_norm,
        TIMESTAMPS_HISTOGRAMS,
        TARGET,
        VALUE,
    )
    fig.show()

### Correlations: Precipitaiton ~ Intensity

In [ ]:
COLOR = "grey"

corr_data = subtarget_statistics_norm.copy()
corr_data.columns = ["_".join(col).strip() for col in corr_data.columns.values]
corr_data = pd.concat([corr_data, df_relevant_norm.precipitation.intensity_hour_shifted], axis=1)
corr_data["n"] = range(len(corr_data))

fig = px.scatter_matrix(
    corr_data,
    dimensions=[
        f"Target-01_{COLOR}",
        f"Target-02_{COLOR}",
        f"Target-03_{COLOR}",
        f"Target-04_{COLOR}",
        f"Target-05_{COLOR}",
        "intensity_hour_shifted",
    ],
    color="n",
)
fig.update_traces(diagonal_visible=False)
fig.update_layout(width=1200, height=1200)
fig.show()

In [ ]:
#! TODO Separate the timesereis in before and after minute 42.

In [ ]:
correlation = subtarget_statistics_norm.corrwith(df_relevant_norm.precipitation.intensity_hour, method="spearman")

# Print the correlation values
print(correlation)

In [ ]:
# # TODO This should be applied to the high temporal data at the start of this NB.

# # Cross-correlation is a measure of similarity between two signals, often used to find features or patterns that appear
# # in both. It can be used to determine how much one series is correlated with a lagged version of another series.
# # In the plot, the x-axis represents the lags and the y-axis represents the cross-correlation values. The error bars
# # represent the confidence intervals. If an error bar does not cross the zero line, it suggests that the correlation at
# # that lag is statistically significant.

# from statsmodels.tsa.stattools import ccf

# # Cross-correlation analysis
# ccf_values = ccf(
#     subtarget_statistics_norm["Target-04"]["white"],
#     df_relevant_norm.precipitation.differential,
#     adjusted=True,
#     alpha=0.05,
# )

# x = list(range(len(subtarget_statistics_norm.index)))
# y = ccf_values[0]
# y_upper = ccf_values[1][:, 1] - y
# y_lower = y - ccf_values[1][:, 0]

# fig = go.Figure(
#     data=go.Scatter(x=x, y=y, error_y=dict(type="data", symmetric=False, array=y_upper, arrayminus=y_lower))
# )

# fig.update_layout(title="Rain Rate and Lidar Intensity Cross-correlation")
# fig.show()